# ============================================================
#   TRUSTWORTHY SENTIMENT ANALYSIS FRAMEWORK
#   Publication-Ready | BiLSTM + Custom Attention | XAI
#   Datasets: IMDb, Amazon, SST-2
# ============================================================

In [1]:
import pandas as pd
import numpy as np
import re
import os
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
 
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, f1_score
)
 
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense,
    Bidirectional, Dropout, Layer
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K
 
import shap
from lime.lime_text import LimeTextExplainer
 
warnings.filterwarnings('ignore')

2026-04-03 18:48:46.104552: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775242126.402522      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775242126.465164      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775242127.036991      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775242127.037028      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775242127.037031      24 computation_placer.cc:177] computation placer alr

# ─────────────────────────────────────────────
#  GLOBAL CONFIG
# ─────────────────────────────────────────────

In [2]:
SAVE_DIR    = "/kaggle/working/plots"
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
 
MAX_FEATURES = 10000
MAX_LEN      = 200
EMBED_DIM    = 128
LSTM_UNITS   = 64
BATCH_SIZE   = 128
EPOCHS       = 10
RANDOM_SEED  = 42
 
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────
#  NLTK SETUP
# ─────────────────────────────────────────────

In [3]:
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.data.find(f'corpora/{pkg}')
    except LookupError:
        nltk.download(pkg, quiet=True)
 
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# ─────────────────────────────────────────────
#  TEXT PREPROCESSING
# ─────────────────────────────────────────────

In [4]:
def preprocess_text(text):
    """Strip HTML, URLs, punctuation → lowercase → lemmatize → remove stopwords."""
    text = re.sub(r'<[^>]+>', '', str(text))          # HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)            # non-alpha
    text = text.lower()
    words = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 1
    ]
    return ' '.join(words)

# ─────────────────────────────────────────────
#  CUSTOM ATTENTION LAYER  (Bahdanau-style)
# ─────────────────────────────────────────────

In [5]:
class Attention(Layer):
    """
    Bahdanau-style additive self-attention.
    Returns (context_vector, attention_weights) when return_weights=True.
 
    Mathematical formulation:
        e_t  = tanh(h_t · W + b)          ← alignment score
        α_t  = softmax(e_t)               ← attention weight
        c    = Σ α_t · h_t                ← context vector
    """
    def __init__(self, return_weights=False, **kwargs):
        self.return_weights = return_weights
        super(Attention, self).__init__(**kwargs)
 
    def build(self, input_shape):
        hidden_dim = input_shape[-1]   # = 2 * LSTM_UNITS (BiLSTM doubles it)
        self.W = self.add_weight(
            name='attn_W',
            shape=(hidden_dim, 1),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attn_b',
            shape=(1,),                # scalar bias, properly broadcast
            initializer='zeros',
            trainable=True
        )
        super(Attention, self).build(input_shape)
 
    def call(self, x):
        # x : (batch, seq_len, hidden_dim)
        e = K.tanh(K.dot(x, self.W) + self.b)    # (batch, seq_len, 1)
        a = K.softmax(e, axis=1)                   # (batch, seq_len, 1)
        context = K.sum(x * a, axis=1)             # (batch, hidden_dim)
        if self.return_weights:
            return context, K.squeeze(a, axis=-1)  # also expose α for visualization
        return context
 
    def get_config(self):
        cfg = super(Attention, self).get_config()
        cfg.update({'return_weights': self.return_weights})
        return cfg
 

# ─────────────────────────────────────────────
#  MODEL BUILDER  (reusable for ablation)
# ─────────────────────────────────────────────

In [6]:
def build_bilstm_model(use_attention=True, lstm_units=LSTM_UNITS):
    """
    Builds BiLSTM model with or without the custom Attention layer.
    Toggle use_attention=False for the ablation baseline.
    """
    inputs  = Input(shape=(MAX_LEN,), name='input')
    x       = Embedding(MAX_FEATURES, EMBED_DIM, input_length=MAX_LEN, name='embedding')(inputs)
    x       = Bidirectional(LSTM(lstm_units, return_sequences=True), name='bilstm')(x)
 
    if use_attention:
        x   = Attention(name='attention')(x)                  # context vector
    else:
        x   = tf.keras.layers.GlobalAveragePooling1D()(x)    # simple pooling baseline
 
    x       = Dense(64, activation='relu', name='dense')(x)
    x       = Dropout(0.5, name='dropout')(x)
    outputs = Dense(1, activation='sigmoid', name='output')(x)
 
    model   = Model(inputs=inputs, outputs=outputs)
    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model
 

# ─────────────────────────────────────────────
#  PLOTTING UTILITIES
# ─────────────────────────────────────────────

In [7]:
def plot_training_curves(history, name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
    axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val',   linewidth=2)
    axes[0].set_title(f'{name} – Accuracy', fontsize=13)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)
 
    axes[1].plot(history.history['loss'],     label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val',   linewidth=2)
    axes[1].set_title(f'{name} – Loss', fontsize=13)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{name}_training_curves.png", dpi=150, bbox_inches='tight')
    plt.close()
 
 
def plot_confusion_matrix(y_true, y_pred, name, labels=('Negative', 'Positive')):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=labels, yticklabels=labels, ax=ax
    )
    ax.set_title(f'{name} – Confusion Matrix', fontsize=13)
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{name}_confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.close()
 
 
def plot_attention_heatmap(tokens, attention_weights, title, filename):
    """Visualise per-token attention weights as a horizontal bar heatmap."""
    # Trim to non-padding tokens
    nonzero = [(t, w) for t, w in zip(tokens, attention_weights) if t != '<PAD>']
    nonzero = nonzero[-30:]   # keep last 30 for readability
    tokens_trimmed  = [x[0] for x in nonzero]
    weights_trimmed = [x[1] for x in nonzero]
 
    fig, ax = plt.subplots(figsize=(8, max(3, len(tokens_trimmed) * 0.4)))
    colors  = plt.cm.YlOrRd(weights_trimmed / (max(weights_trimmed) + 1e-9))
    bars    = ax.barh(range(len(tokens_trimmed)), weights_trimmed, color=colors)
    ax.set_yticks(range(len(tokens_trimmed)))
    ax.set_yticklabels(tokens_trimmed, fontsize=9)
    ax.set_xlabel('Attention Weight')
    ax.set_title(title, fontsize=12)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{filename}", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved attention heatmap → {filename}")
 
 
def plot_results_comparison(results_df):
    """Side-by-side bar chart comparing LR vs BiLSTM+Attention across datasets."""
    x    = np.arange(len(results_df))
    w    = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, results_df['LR_Accuracy'],           w, label='Logistic Regression', color='#5B9BD5')
    ax.bar(x + w/2, results_df['BiLSTM_Attn_Accuracy'],  w, label='BiLSTM + Attention',  color='#ED7D31')
    ax.set_xticks(x)
    ax.set_xticklabels(results_df['Dataset'], fontsize=12)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.7, 1.0)
    ax.set_title('Model Comparison Across Datasets', fontsize=14)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/model_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
 
 
def plot_ablation(ablation_df):
    """Bar chart for ablation study results."""
    fig, ax = plt.subplots(figsize=(8, 4))
    colors  = ['#C00000' if 'Without' in c else '#70AD47' for c in ablation_df['Config']]
    ax.barh(ablation_df['Config'], ablation_df['Val_Accuracy'], color=colors)
    ax.set_xlabel('Validation Accuracy')
    ax.set_title('Ablation Study – Effect of Attention Layer', fontsize=13)
    ax.set_xlim(0.75, 1.0)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/ablation_study.png", dpi=150, bbox_inches='tight')
    plt.close()
 

# ─────────────────────────────────────────────
#  ATTENTION WEIGHT EXTRACTOR
# ─────────────────────────────────────────────

In [8]:
def build_attention_extractor(trained_model):
    """
    Creates a sub-model that outputs attention weights alongside predictions.
    Works by building a new Model that shares weights with the trained model.
    """
    attn_layer   = trained_model.get_layer('attention')
    attn_layer.return_weights = True    # flip the flag
 
    inp          = trained_model.input
    bilstm_out   = trained_model.get_layer('bilstm').output
    ctx, weights = attn_layer(bilstm_out)   # now returns (context, weights)
 
    extractor = Model(inputs=inp, outputs=weights)
    return extractor
 

# ─────────────────────────────────────────────
#  MAIN TRAINING PIPELINE
# ─────────────────────────────────────────────

In [9]:
all_results = []   # will collect cross-dataset summary
 
def train_models(df, dataset_name):
    print(f"\n{'='*60}")
    print(f"  Training on: {dataset_name}")
    print(f"{'='*60}")
 
    X = df['cleaned']
    y = df['sentiment']
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
 
    # ── 1. Baseline: TF-IDF + Logistic Regression ─────────────
    print("\n[1/4] Training Logistic Regression baseline...")
    vectorizer       = TfidfVectorizer(max_features=MAX_FEATURES)
    X_train_tfidf    = vectorizer.fit_transform(X_train)
    X_test_tfidf     = vectorizer.transform(X_test)
 
    lr               = LogisticRegression(max_iter=500, random_state=RANDOM_SEED)
    lr.fit(X_train_tfidf, y_train)
    lr_pred          = lr.predict(X_test_tfidf)
    lr_acc           = accuracy_score(y_test, lr_pred)
    lr_f1            = f1_score(y_test, lr_pred, average='weighted')
    print(f"   LR Accuracy: {lr_acc:.4f}  |  F1: {lr_f1:.4f}")
 
    # ── 2. Tokenisation ────────────────────────────────────────
    tokenizer        = Tokenizer(num_words=MAX_FEATURES, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)
 
    X_train_seq      = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
    X_test_seq       = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')
 
    # ── 3. BiLSTM + Attention ──────────────────────────────────
    print("\n[2/4] Training BiLSTM + Attention model...")
    model            = build_bilstm_model(use_attention=True)
    model.summary()
 
    es               = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    history          = model.fit(
        X_train_seq, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[es],
        verbose=1
    )
 
    # Evaluation
    _, bilstm_acc    = model.evaluate(X_test_seq, y_test, verbose=0)
    y_pred_probs     = model.predict(X_test_seq, verbose=0)
    y_pred_classes   = (y_pred_probs > 0.5).astype(int)
    bilstm_f1        = f1_score(y_test, y_pred_classes, average='weighted')
 
    print(f"\n--- Full Classification Report: {dataset_name} ---")
    print(classification_report(y_test, y_pred_classes, target_names=['Negative','Positive']))
 
    # ── 4. Save plots ──────────────────────────────────────────
    print("[3/4] Saving plots...")
    plot_training_curves(history, dataset_name)
    plot_confusion_matrix(y_test, y_pred_classes, dataset_name)
 
    # ── 5. Attention visualisation ─────────────────────────────
    print("[4/4] Visualising attention weights...")
    try:
        extractor  = build_attention_extractor(model)
        sample_seq = X_test_seq[:3]
        attn_wts   = extractor.predict(sample_seq, verbose=0)   # (3, MAX_LEN)
 
        idx_to_word = {v: k for k, v in tokenizer.word_index.items()}
 
        for i in range(3):
            raw_ids = sample_seq[i]
            tokens  = [idx_to_word.get(idx, '<PAD>') for idx in raw_ids]
            weights = attn_wts[i]
            plot_attention_heatmap(
                tokens, weights,
                title=f'{dataset_name} – Sample {i+1} Attention Heatmap',
                filename=f'{dataset_name}_attention_sample{i+1}.png'
            )
    except Exception as e:
        print(f"  [WARN] Attention extractor skipped: {e}")
 
    # ── Collect results for comparison table ───────────────────
    all_results.append({
        'Dataset':              dataset_name,
        'LR_Accuracy':          round(lr_acc, 4),
        'LR_F1':                round(lr_f1, 4),
        'BiLSTM_Attn_Accuracy': round(bilstm_acc, 4),
        'BiLSTM_Attn_F1':       round(bilstm_f1, 4),
    })
 
    return model, tokenizer, vectorizer, lr, X_train, X_test, X_test_seq, X_test_tfidf, y_test
 

# ─────────────────────────────────────────────
#  K-FOLD CROSS VALIDATION
# ─────────────────────────────────────────────

In [10]:
def kfold_evaluation(df, dataset_name, n_splits=5):
    """
    5-fold stratified cross-validation on the BiLSTM+Attention model.
    Reports mean ± std accuracy and F1 — required for statistical validity.
    """
    print(f"\n[K-Fold CV] Running {n_splits}-fold CV on {dataset_name}...")
    X = df['cleaned'].values
    y = df['sentiment'].values
 
    skf      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_acc = []
    fold_f1  = []
 
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
 
        tok = Tokenizer(num_words=MAX_FEATURES, oov_token='<OOV>')
        tok.fit_on_texts(X_tr)
        X_tr_seq  = pad_sequences(tok.texts_to_sequences(X_tr),  maxlen=MAX_LEN, padding='post')
        X_val_seq = pad_sequences(tok.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post')
 
        m = build_bilstm_model(use_attention=True)
        m.fit(
            X_tr_seq, y_tr,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_val_seq, y_val),
            callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
            verbose=0
        )
        preds = (m.predict(X_val_seq, verbose=0) > 0.5).astype(int)
        fold_acc.append(accuracy_score(y_val, preds))
        fold_f1.append(f1_score(y_val, preds, average='weighted'))
        print(f"  Fold {fold}: Acc={fold_acc[-1]:.4f}  F1={fold_f1[-1]:.4f}")
 
        K.clear_session()   # free GPU memory between folds
 
    print(f"\n  {dataset_name} K-Fold Results:")
    print(f"  Accuracy : {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
    print(f"  F1-Score : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
    return {
        'Dataset':   dataset_name,
        'CV_Acc_Mean': round(np.mean(fold_acc), 4),
        'CV_Acc_Std':  round(np.std(fold_acc),  4),
        'CV_F1_Mean':  round(np.mean(fold_f1),  4),
        'CV_F1_Std':   round(np.std(fold_f1),   4),
    }
 

# ─────────────────────────────────────────────
#  ABLATION STUDY
# ─────────────────────────────────────────────

In [11]:
def ablation_study(X_train_seq, X_test_seq, y_train, y_test, dataset_name):
    """
    Compares four configurations to justify architectural choices:
      1. BiLSTM (32 units) + Attention
      2. BiLSTM (64 units) – No Attention   ← key baseline
      3. BiLSTM (64 units) + Attention      ← proposed model
      4. BiLSTM (128 units) + Attention
    """
    print(f"\n[Ablation Study] Running on {dataset_name}...")
 
    configs = [
        {'label': 'BiLSTM-32 + Attention',    'lstm': 32,  'attention': True},
        {'label': 'BiLSTM-64 Without Attention','lstm': 64, 'attention': False},
        {'label': 'BiLSTM-64 + Attention',     'lstm': 64,  'attention': True},
        {'label': 'BiLSTM-128 + Attention',    'lstm': 128, 'attention': True},
    ]
 
    rows = []
    for cfg in configs:
        m = build_bilstm_model(use_attention=cfg['attention'], lstm_units=cfg['lstm'])
        m.fit(
            X_train_seq, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.2,
            callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
            verbose=0
        )
        _, val_acc = m.evaluate(X_test_seq, y_test, verbose=0)
        preds      = (m.predict(X_test_seq, verbose=0) > 0.5).astype(int)
        val_f1     = f1_score(y_test, preds, average='weighted')
        rows.append({'Config': cfg['label'], 'Val_Accuracy': round(val_acc, 4), 'Val_F1': round(val_f1, 4)})
        print(f"  {cfg['label']:<35} Acc={val_acc:.4f}  F1={val_f1:.4f}")
        K.clear_session()
 
    abl_df = pd.DataFrame(rows)
    abl_df.to_csv(f"{RESULTS_DIR}/{dataset_name}_ablation.csv", index=False)
    plot_ablation(abl_df)
    print(f"  ✓ Ablation results saved.")
    return abl_df
 

# ─────────────────────────────────────────────
#  XAI MODULE — SHAP
# ─────────────────────────────────────────────

In [12]:
def run_shap(lr_model, vectorizer, X_test_tfidf, feature_names, dataset_name, n_samples=100):
    """
    Uses KernelExplainer on the LR model (model-agnostic, publication-standard).
    Saves both a summary plot and a bar plot of top features.
    """
    print(f"\n[SHAP] Generating explanations for {dataset_name}...")
    try:
        background   = shap.sample(X_test_tfidf, 50)
        explainer    = shap.KernelExplainer(lr_model.predict_proba, background)
        shap_values  = explainer.shap_values(X_test_tfidf[:n_samples], nsamples=100)
 
        # Summary plot (beeswarm)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(
            shap_values[1], X_test_tfidf[:n_samples],
            feature_names=feature_names,
            max_display=20,
            show=False
        )
        plt.title(f'{dataset_name} – SHAP Summary (Positive Class)', fontsize=13)
        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/{dataset_name}_shap_summary.png", dpi=150, bbox_inches='tight')
        plt.close()
 
        # Bar plot (mean |SHAP|)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(
            shap_values[1], X_test_tfidf[:n_samples],
            feature_names=feature_names,
            plot_type='bar',
            max_display=20,
            show=False
        )
        plt.title(f'{dataset_name} – SHAP Feature Importance', fontsize=13)
        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/{dataset_name}_shap_bar.png", dpi=150, bbox_inches='tight')
        plt.close()
 
        print(f"  ✓ SHAP plots saved for {dataset_name}.")
    except Exception as e:
        print(f"  [WARN] SHAP skipped for {dataset_name}: {e}")
 

# ─────────────────────────────────────────────
#  XAI MODULE — LIME
# ─────────────────────────────────────────────

In [13]:
def run_lime(bilstm_model, tokenizer, X_test_raw, dataset_name, n_samples=3):
    """
    Applies LIME to the BiLSTM model to explain individual predictions.
    Saves one PNG per sample.
    """
    print(f"\n[LIME] Generating explanations for {dataset_name}...")
 
    def predict_proba(texts):
        seqs  = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN, padding='post')
        probs = bilstm_model.predict(seqs, verbose=0)
        return np.hstack([1 - probs, probs])
 
    lime_exp = LimeTextExplainer(class_names=['Negative', 'Positive'])
 
    for i in range(min(n_samples, len(X_test_raw))):
        try:
            exp = lime_exp.explain_instance(
                X_test_raw.iloc[i],
                predict_proba,
                num_features=15,
                num_samples=500
            )
            fig = exp.as_pyplot_figure()
            fig.suptitle(f'{dataset_name} – LIME Explanation (Sample {i+1})', fontsize=11)
            fig.savefig(f"{SAVE_DIR}/{dataset_name}_lime_sample{i+1}.png", dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f"  ✓ LIME sample {i+1} saved.")
        except Exception as e:
            print(f"  [WARN] LIME sample {i+1} failed: {e}")
 
 

# ─────────────────────────────────────────────
#  ROBUSTNESS TESTING MODULE
# ─────────────────────────────────────────────

In [14]:
ROBUSTNESS_PAIRS = [
    # (original_text, perturbed_text, perturbation_type)
    (
        "This movie was absolutely fantastic and the storyline was engaging from start to finish.",
        "This movie was absolutely okay and the storyline was fine from start to finish.",
        "Positive Adjective Weakening"
    ),
    (
        "The product quality is excellent and I highly recommend it to everyone.",
        "The product quality is not excellent and I do not recommend it to everyone.",
        "Negation Injection"
    ),
    (
        "Truly an outstanding experience, I loved every moment of it.",
        "Truly an outstaNding expErience, I lovd every momnt of it.",
        "Typo / Noise Injection"
    ),
    (
        "Brilliant performance by the cast, visually stunning and emotionally powerful.",
        "Decent performance by the cast, visually acceptable and somewhat emotional.",
        "Synonym Substitution (Weaker)"
    ),
    (
        "A masterpiece of cinema that left me speechless.",
        "A mediocre piece of cinema that left me indifferent.",
        "Semantic Reversal"
    ),
]
 
def run_robustness_tests(bilstm_model, tokenizer, dataset_name):
    """
    Tests the model on curated original/perturbed text pairs.
    Saves a summary bar chart and CSV.
    """
    print(f"\n[Robustness] Running robustness tests for {dataset_name}...")
 
    def get_confidence(texts):
        seqs  = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN, padding='post')
        probs = bilstm_model.predict(seqs, verbose=0).flatten()
        return probs * 100
 
    records = []
    for orig, alt, ptype in ROBUSTNESS_PAIRS:
        orig_conf = get_confidence([orig])[0]
        alt_conf  = get_confidence([alt])[0]
        drop      = orig_conf - alt_conf
        records.append({
            'Perturbation':          ptype,
            'Original_Confidence_%': round(orig_conf, 2),
            'Altered_Confidence_%':  round(alt_conf,  2),
            'Confidence_Drop_%':     round(drop,       2),
        })
        print(f"  [{ptype}]")
        print(f"    Original → {orig_conf:.1f}%   Altered → {alt_conf:.1f}%   Δ = {drop:.1f}%")
 
    rob_df = pd.DataFrame(records)
    rob_df.to_csv(f"{RESULTS_DIR}/{dataset_name}_robustness.csv", index=False)
 
    # Bar chart of confidence drops
    fig, ax = plt.subplots(figsize=(10, 4))
    colors  = ['#C00000' if d > 0 else '#70AD47' for d in rob_df['Confidence_Drop_%']]
    ax.bar(rob_df['Perturbation'], rob_df['Confidence_Drop_%'], color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Confidence Drop (%)')
    ax.set_title(f'{dataset_name} – Robustness Test: Confidence Drop Under Perturbation', fontsize=12)
    plt.xticks(rotation=20, ha='right', fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{dataset_name}_robustness.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Robustness report saved.")
    return rob_df
 

 
# ─────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────

In [15]:
print("\n" + "="*60)
print("  LOADING DATASETS")
print("="*60)
 
# ── IMDb ───────────────────────────────────────────────────────
print("\nLoading IMDb...")
df_imdb = pd.read_csv(
    '/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'
)
df_imdb['sentiment'] = df_imdb['sentiment'].map({'positive': 1, 'negative': 0})
df_imdb['cleaned']   = df_imdb['review'].apply(preprocess_text)
print(f"  IMDb shape: {df_imdb.shape}")
 
# ── Amazon Product Reviews ─────────────────────────────────────
# Expected columns: 'reviewText' and 'overall' (1–5 star rating)
# Binarise: rating >= 4 → positive (1), rating <= 2 → negative (0)
print("Loading Amazon Reviews...")
import bz2

def load_amazon_reviews(path, n_samples=25000):
    texts = []
    labels = []
    
    with bz2.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            # Format is "__label__2 This is a great product"
            label = 1 if line.startswith('__label__2') else 0
            text = line.split(' ', 1)[1].strip()
            
            texts.append(text)
            labels.append(label)
            
            if len(texts) >= n_samples * 2: # Stop once we have enough for balancing
                break
                
    return pd.DataFrame({'reviewText': texts, 'sentiment': labels})

print("Loading Amazon Reviews...")
# Use the function to load and then preprocess
df_amazon = load_amazon_reviews('/kaggle/input/datasets/bittlingmayer/amazonreviews/train.ft.txt.bz2')
df_amazon['cleaned'] = df_amazon['reviewText'].apply(preprocess_text)

# Balancing (optional since we're manually loading, but keeps your pipeline consistent)
df_amazon = (df_amazon.groupby('sentiment')
             .apply(lambda g: g.sample(min(len(g), 25000), random_state=RANDOM_SEED))
             .reset_index(drop=True))

print(f"  Amazon shape: {df_amazon.shape}")
 
# ── SST-2 ──────────────────────────────────────────────────────
# Expected columns: 'sentence' and 'label' (0 or 1)
print("Loading SST-2...")
df_sst2 = pd.read_csv(
    '/kaggle/input/datasets/subhranilwayne/stanford-ss-2/train.csv'
)
df_sst2.rename(columns={'sentence': 'text', 'label': 'sentiment'}, inplace=True)
df_sst2['sentiment'] = df_sst2['sentiment'].astype(int)
df_sst2['cleaned']   = df_sst2['text'].apply(preprocess_text)
print(f"  SST-2 shape: {df_sst2.shape}")
 


  LOADING DATASETS

Loading IMDb...
  IMDb shape: (50000, 3)
Loading Amazon Reviews...
Loading Amazon Reviews...
  Amazon shape: (49494, 3)
Loading SST-2...
  SST-2 shape: (6920, 3)


# ─────────────────────────────────────────────
#  TRAINING — ALL THREE DATASETS
# ─────────────────────────────────────────────

In [16]:
datasets = [
    (df_imdb,   "IMDB"),
    (df_amazon, "Amazon"),
    (df_sst2,   "SST2"),
]
 
trained   = {}   # store artefacts per dataset for XAI reuse
cv_results = []
 
for df, name in datasets:
    (model, tokenizer, vectorizer, lr_model,
     X_train, X_test, X_test_seq, X_test_tfidf, y_test) = train_models(df, name)
 
    trained[name] = {
        'model':        model,
        'tokenizer':    tokenizer,
        'vectorizer':   vectorizer,
        'lr_model':     lr_model,
        'X_train':      X_train,
        'X_test':       X_test,
        'X_test_seq':   X_test_seq,
        'X_test_tfidf': X_test_tfidf,
        'y_test':       y_test,
    }


  Training on: IMDB

[1/4] Training Logistic Regression baseline...
   LR Accuracy: 0.8905  |  F1: 0.8905

[2/4] Training BiLSTM + Attention model...


I0000 00:00:1775242202.414811      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775242202.420670      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ (None, 128)            │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,266 (5.29 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ (None, 128)            │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,266 (5.29 MB)

 Trainable params: 1,387,266 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.5261 - loss: 0.6934 - val_accuracy: 0.5217 - val_loss: 0.6927
Epoch 2/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5183 - loss: 0.6937 - val_accuracy: 0.5217 - val_loss: 0.6924
Epoch 3/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5290 - loss: 0.6926 - val_accuracy: 0.5217 - val_loss: 0.6922
Epoch 4/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5291 - loss: 0.6914 - val_accuracy: 0.5217 - val_loss: 0.6924
Epoch 5/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5281 - loss: 0.6924 - val_accuracy: 0.5217 - val_loss: 0.6920
Epoch 6/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5253 - loss: 0.6924 - val_accuracy: 0.5217 - val_loss: 0.6914
Epoch 7/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5359 - loss: 0.6888 - val_accuracy: 0.6462 - val_loss: 0.6614
Epoch 8/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7649 - loss: 0.5473 - val_accuracy: 0.6832 - v

# ─────────────────────────────────────────────
#  CROSS-DATASET RESULTS TABLE
# ─────────────────────────────────────────────

In [17]:
print("\n" + "="*60)
print("  CROSS-DATASET RESULTS SUMMARY")
print("="*60)
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
results_df.to_csv(f"{RESULTS_DIR}/cross_dataset_results.csv", index=False)
plot_results_comparison(results_df)
print("\n✓ Results table saved → cross_dataset_results.csv")


  CROSS-DATASET RESULTS SUMMARY
Dataset  LR_Accuracy  LR_F1  BiLSTM_Attn_Accuracy  BiLSTM_Attn_F1
   IMDB       0.8905 0.8905                0.8798          0.8797
 Amazon       0.8825 0.8825                0.8765          0.8764
   SST2       0.7608 0.7601                0.6315          0.6191

✓ Results table saved → cross_dataset_results.csv


# ─────────────────────────────────────────────
#  K-FOLD CROSS VALIDATION
# ─────────────────────────────────────────────

In [18]:
print("\n" + "="*60)
print("  K-FOLD CROSS VALIDATION (5-fold)")
print("="*60)
# NOTE: CV on all 3 datasets is compute-heavy.
# Run on IMDb by default; replicate for others on a GPU node.
cv_result = kfold_evaluation(df_imdb, "IMDB", n_splits=5)
cv_results.append(cv_result)
 
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(f"{RESULTS_DIR}/kfold_results.csv", index=False)
print("\n✓ K-Fold results saved → kfold_results.csv")


  K-FOLD CROSS VALIDATION (5-fold)

[K-Fold CV] Running 5-fold CV on IMDB...
  Fold 1: Acc=0.8824  F1=0.8824
  Fold 2: Acc=0.8860  F1=0.8859
  Fold 3: Acc=0.8682  F1=0.8677
  Fold 4: Acc=0.8863  F1=0.8863
  Fold 5: Acc=0.8851  F1=0.8850

  IMDB K-Fold Results:
  Accuracy : 0.8816 ± 0.0068
  F1-Score : 0.8814 ± 0.0070

✓ K-Fold results saved → kfold_results.csv


# ─────────────────────────────────────────────
#  ABLATION STUDY
# ─────────────────────────────────────────────

In [19]:
print("\n" + "=" * 60)
print("  ABLATION STUDY")
print("=" * 60)

# 1. Get the trained IMDb artifacts
t = trained["IMDB"]

# 2. Re-generate the training sequences to ensure they are clean NumPy arrays
# X_train is the raw text from our 'trained' dictionary
tokenizer = t['tokenizer']
X_train_seq_abl = pad_sequences(tokenizer.texts_to_sequences(t['X_train']), 
                                maxlen=MAX_LEN, padding='post')

# 3. Ensure y_train is a clean NumPy array of integers
# We use .loc and .values to get a clean array from the original dataframe
train_idx = t['X_train'].index
y_train_numeric = df_imdb.loc[train_idx, 'sentiment'].values.astype('int32')

# 4. Run the study
# Ensure X_test_seq and y_test are also clean arrays
abl_df = ablation_study(
    np.array(X_train_seq_abl), 
    np.array(t['X_test_seq']), 
    y_train_numeric, 
    np.array(t['y_test']), 
    "IMDB"
)


  ABLATION STUDY

[Ablation Study] Running on IMDB...
  BiLSTM-32 + Attention               Acc=0.8820  F1=0.8820
  BiLSTM-64 Without Attention         Acc=0.8810  F1=0.8810
  BiLSTM-64 + Attention               Acc=0.8818  F1=0.8817
  BiLSTM-128 + Attention              Acc=0.8856  F1=0.8856
  ✓ Ablation results saved.


# ─────────────────────────────────────────────
#  XAI — SHAP & LIME (all datasets)
# ─────────────────────────────────────────────

In [20]:
print("\n" + "="*60)
print("  XAI MODULE: SHAP + LIME")
print("="*60)
 
for name, art in trained.items():
    feature_names = art['vectorizer'].get_feature_names_out()
    run_shap(art['lr_model'], art['vectorizer'],
             art['X_test_tfidf'], feature_names, name)
    run_lime(art['model'], art['tokenizer'], art['X_test'], name)


  XAI MODULE: SHAP + LIME

[SHAP] Generating explanations for IMDB...


  0%|          | 0/100 [00:00<?, ?it/s]

  [WARN] SHAP skipped for IMDB: The shape of the shap_values matrix does not match the shape of the provided data matrix.

[LIME] Generating explanations for IMDB...
  ✓ LIME sample 1 saved.
  ✓ LIME sample 2 saved.
  ✓ LIME sample 3 saved.

[SHAP] Generating explanations for Amazon...


  0%|          | 0/100 [00:00<?, ?it/s]

  [WARN] SHAP skipped for Amazon: The shape of the shap_values matrix does not match the shape of the provided data matrix.

[LIME] Generating explanations for Amazon...
  ✓ LIME sample 1 saved.
  ✓ LIME sample 2 saved.
  ✓ LIME sample 3 saved.

[SHAP] Generating explanations for SST2...


  0%|          | 0/100 [00:00<?, ?it/s]

  [WARN] SHAP skipped for SST2: The shape of the shap_values matrix does not match the shape of the provided data matrix.

[LIME] Generating explanations for SST2...
  ✓ LIME sample 1 saved.
  ✓ LIME sample 2 saved.
  ✓ LIME sample 3 saved.


# ─────────────────────────────────────────────
#  ROBUSTNESS TESTS (all datasets)
# ─────────────────────────────────────────────

In [21]:
print("\n" + "="*60)
print("  ROBUSTNESS TESTING MODULE")
print("="*60)
 
rob_results = {}
for name, art in trained.items():
    rob_results[name] = run_robustness_tests(art['model'], art['tokenizer'], name)
 


  ROBUSTNESS TESTING MODULE

[Robustness] Running robustness tests for IMDB...
  [Positive Adjective Weakening]
    Original → 61.9%   Altered → 27.0%   Δ = 34.9%
  [Negation Injection]
    Original → 84.8%   Altered → 68.5%   Δ = 16.3%
  [Typo / Noise Injection]
    Original → 76.4%   Altered → 58.6%   Δ = 17.8%
  [Synonym Substitution (Weaker)]
    Original → 71.3%   Altered → 45.7%   Δ = 25.6%
  [Semantic Reversal]
    Original → 56.7%   Altered → 17.2%   Δ = 39.5%
  ✓ Robustness report saved.

[Robustness] Running robustness tests for Amazon...
  [Positive Adjective Weakening]
    Original → 65.2%   Altered → 15.3%   Δ = 49.9%
  [Negation Injection]
    Original → 95.8%   Altered → 90.1%   Δ = 5.7%
  [Typo / Noise Injection]
    Original → 96.2%   Altered → 92.2%   Δ = 4.0%
  [Synonym Substitution (Weaker)]
    Original → 93.3%   Altered → 7.9%   Δ = 85.5%
  [Semantic Reversal]
    Original → 86.1%   Altered → 3.6%   Δ = 82.5%
  ✓ Robustness report saved.

[Robustness] Running rob

# ─────────────────────────────────────────────
#  FINAL SUMMARY PRINTOUT
# ─────────────────────────────────────────────

In [22]:
print("\n" + "="*60)
print("  FINAL SUMMARY")
print("="*60)
print("\nCross-Dataset Accuracy & F1:")
print(results_df.to_string(index=False))
 
if cv_results:
    print("\n5-Fold CV (IMDb):")
    print(cv_df.to_string(index=False))
 
print("\nAblation Study (IMDb):")
print(abl_df.to_string(index=False))
 
print(f"\n✓ All plots saved in  : {SAVE_DIR}")
print(f"✓ All CSVs saved in   : {RESULTS_DIR}")
print("\n[DONE] Framework execution complete.\n")


  FINAL SUMMARY

Cross-Dataset Accuracy & F1:
Dataset  LR_Accuracy  LR_F1  BiLSTM_Attn_Accuracy  BiLSTM_Attn_F1
   IMDB       0.8905 0.8905                0.8798          0.8797
 Amazon       0.8825 0.8825                0.8765          0.8764
   SST2       0.7608 0.7601                0.6315          0.6191

5-Fold CV (IMDb):
Dataset  CV_Acc_Mean  CV_Acc_Std  CV_F1_Mean  CV_F1_Std
   IMDB       0.8816      0.0068      0.8814      0.007

Ablation Study (IMDb):
                     Config  Val_Accuracy  Val_F1
      BiLSTM-32 + Attention        0.8820  0.8820
BiLSTM-64 Without Attention        0.8810  0.8810
      BiLSTM-64 + Attention        0.8818  0.8817
     BiLSTM-128 + Attention        0.8856  0.8856

✓ All plots saved in  : /kaggle/working/plots
✓ All CSVs saved in   : /kaggle/working/results

[DONE] Framework execution complete.

